<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-05-bigquery-ml/lesson-5.2-time-series/practice/GCP_Capstone_5.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 5.2 — Time Series & Anomaly Detection

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Authenticate with Application Default Credentials, connect the BigQuery client, and generate 30 days of synthetic hourly metrics. Run this once, top-to-bottom, before any exercise below. Change `PROJECT_ID` to your own project.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
USD_INR = 85  # for any INR cost display

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# Ensure this module's datasets exist (idempotent -- BigQuery never auto-creates them).
# NOTE: cells that read rag_data.document_features need Lesson 5.1 run first.
for _ds in ('rag_data', 'ml_models'):
    _d = bigquery.Dataset(f'{PROJECT_ID}.{_ds}'); _d.location = 'US'
    client.create_dataset(_d, exists_ok=True)

def run_query(sql):
    return client.query(sql).to_dataframe()

def run_ddl(sql):
    job = client.query(sql)
    job.result()
    print(f'Done: {job.num_dml_affected_rows or "OK"}')

print(f'Connected to {PROJECT_ID}')

In [ ]:
# Generate 30 days of hourly query + cost data with realistic patterns
# (daily seasonality, weekend dip, upward trend, noise, rare spikes)
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.hourly_metrics` AS
WITH hours AS (
  SELECT ts
  FROM UNNEST(GENERATE_TIMESTAMP_ARRAY(
    TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY),
    CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)) AS ts
)
SELECT
  ts AS hour_bucket,
  CAST(
    50  -- base
    + 30 * SIN(2 * ACOS(-1) * EXTRACT(HOUR FROM ts) / 24)  -- daily pattern
    + 15 * IF(EXTRACT(DAYOFWEEK FROM ts) IN (1, 7), -1, 1)  -- weekend dip
    + 0.5 * TIMESTAMP_DIFF(ts, TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY), HOUR) / 24  -- trend
    + 10 * (RAND() - 0.5)  -- noise
    + IF(RAND() < 0.02, 80, 0)  -- rare spikes
  AS INT64) AS query_count,
  ROUND(
    0.003 * (
      50 + 30 * SIN(2 * ACOS(-1) * EXTRACT(HOUR FROM ts) / 24)
      + 10 * (RAND() - 0.5)
      + IF(RAND() < 0.01, 50, 0)
    ), 4) AS hourly_cost
FROM hours
''')
print('30 days of hourly data generated')
print(run_query(f'SELECT * FROM `{PROJECT_ID}.rag_data.hourly_metrics` ORDER BY hour_bucket DESC LIMIT 5'))

## Exercise 1: First ARIMA_PLUS Model

**Difficulty:** Easy

Train ARIMA_PLUS on hourly query counts. Check ML.ARIMA_EVALUATE for selected (p,d,q).

1. CREATE MODEL with ARIMA_PLUS
2. Set timestamp, data, frequency columns
3. Query ML.ARIMA_EVALUATE

In [ ]:
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.query_forecast`
OPTIONS (
  model_type = 'ARIMA_PLUS',
  time_series_timestamp_col = 'hour_bucket',
  time_series_data_col = 'query_count',
  auto_arima = TRUE,
  data_frequency = 'HOURLY',
  decompose_time_series = TRUE,
  clean_spikes_and_dips = TRUE,
  horizon = 168,
  forecast_limit_lower_bound = 0
) AS
SELECT hour_bucket, query_count
FROM `{PROJECT_ID}.rag_data.hourly_metrics`
''')
print('Query forecast model trained')

# auto_arima picked (p,d,q) for us — inspect what it chose, plus AIC
print('\n=== ARIMA Evaluation ===')
print(run_query(f'SELECT * FROM ML.ARIMA_EVALUATE(MODEL `{PROJECT_ID}.ml_models.query_forecast`)'))

## Exercise 2: ML.FORECAST 72 Hours

**Difficulty:** Easy

Forecast next 72 hours. Print forecast_value + confidence intervals.

1. Call ML.FORECAST with horizon=72
2. Set confidence_level=0.95
3. Print all output columns

In [ ]:
forecast = run_query(f'''
SELECT
  forecast_timestamp,
  ROUND(forecast_value, 1) AS predicted_queries,
  ROUND(prediction_interval_lower_bound, 1) AS lower_95,
  ROUND(prediction_interval_upper_bound, 1) AS upper_95
FROM ML.FORECAST(
  MODEL `{PROJECT_ID}.ml_models.query_forecast`,
  STRUCT(72 AS horizon, 0.95 AS confidence_level))
ORDER BY forecast_timestamp
''')
print(f'Forecast: {len(forecast)} rows')
print(forecast.head(10))

## Exercise 3: Capacity Planning Query

**Difficulty:** Easy

Find forecast hours where upper_bound exceeds a threshold. Alert for auto-scaling.

1. Filter ML.FORECAST WHERE upper_bound > threshold
2. Print timestamp and worst-case values
3. Count how many hours need scaling

In [ ]:
# Reuse the 72-hour forecast from Exercise 2. Any hour whose worst-case (upper_95)
# exceeds our per-hour capacity of 100 queries needs auto-scaling headroom.
CAPACITY_THRESHOLD = 100

peaks = forecast[forecast['upper_95'] > CAPACITY_THRESHOLD]
print(f'Hours exceeding {CAPACITY_THRESHOLD} queries (worst case): {len(peaks)}')
print(peaks[['forecast_timestamp', 'predicted_queries', 'upper_95']])

## Exercise 4: Cost Anomaly Detection

**Difficulty:** Medium

Train on hourly cost. Run ML.DETECT_ANOMALIES. Print flagged hours with probabilities.

1. CREATE MODEL on hourly_cost column
2. ML.DETECT_ANOMALIES with threshold=0.95
3. Filter WHERE is_anomaly = TRUE

In [ ]:
# Train a second ARIMA_PLUS model, this time on hourly_cost
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.cost_anomaly`
OPTIONS (
  model_type = 'ARIMA_PLUS',
  time_series_timestamp_col = 'hour_bucket',
  time_series_data_col = 'hourly_cost',
  decompose_time_series = TRUE,
  clean_spikes_and_dips = TRUE
) AS
SELECT hour_bucket, hourly_cost
FROM `{PROJECT_ID}.rag_data.hourly_metrics`
''')
print('Cost anomaly model trained')

# Anomalies = points outside the model's expected band at 95% probability
anomalies = run_query(f'''
SELECT
  hour_bucket,
  hourly_cost,
  is_anomaly,
  ROUND(anomaly_probability, 3) AS anomaly_prob,
  ROUND(lower_bound, 4) AS expected_low,
  ROUND(upper_bound, 4) AS expected_high
FROM ML.DETECT_ANOMALIES(
  MODEL `{PROJECT_ID}.ml_models.cost_anomaly`,
  STRUCT(0.95 AS anomaly_prob_threshold))
WHERE is_anomaly = TRUE
ORDER BY anomaly_probability DESC
''')
print(f'\nAnomalies found: {len(anomalies)}')
print(anomalies.head(10))

## Exercise 5: Anomalies on New Data

**Difficulty:** Medium

Pass today's data to ML.DETECT_ANOMALIES as third argument. Flag real-time issues.

1. Write subquery for today's hourly metrics
2. Pass as third argument to ML.DETECT_ANOMALIES
3. Check for is_anomaly = TRUE

In [ ]:
# The 3rd argument feeds NEW rows to score against the trained model —
# this is how you check today's live metrics against learned patterns.
new_anomalies = run_query(f'''
SELECT *
FROM ML.DETECT_ANOMALIES(
  MODEL `{PROJECT_ID}.ml_models.cost_anomaly`,
  STRUCT(0.90 AS anomaly_prob_threshold),
  (SELECT hour_bucket, hourly_cost
   FROM `{PROJECT_ID}.rag_data.hourly_metrics`
   WHERE hour_bucket >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR))
)
WHERE is_anomaly = TRUE
''')
print(f'Real-time anomalies in last 24h: {len(new_anomalies)}')
if len(new_anomalies) > 0:
    print(new_anomalies)

## Exercise 6: ML.EXPLAIN_FORECAST

**Difficulty:** Medium

Decompose query patterns. Interpret trend, daily, weekly components.

1. Run ML.EXPLAIN_FORECAST
2. Separate history vs forecast rows
3. Explain what daily and weekly patterns mean for DocuMind

In [ ]:
decomp = run_query(f'''
SELECT
  time_series_type,
  time_series_timestamp,
  ROUND(time_series_data, 1) AS value,
  ROUND(trend, 1) AS trend,
  ROUND(seasonal_period_daily, 1) AS daily_pattern,
  ROUND(seasonal_period_weekly, 1) AS weekly_pattern,
  ROUND(spikes_and_dips, 1) AS spikes,
  ROUND(step_changes, 1) AS step_changes
FROM ML.EXPLAIN_FORECAST(
  MODEL `{PROJECT_ID}.ml_models.query_forecast`,
  STRUCT(48 AS horizon))
ORDER BY time_series_timestamp
''')

print('=== Forecast decomposition ===')
print(decomp[decomp['time_series_type'] == 'forecast'].head(10))

# Recent history for comparison
print('\n=== Recent history ===')
hist = decomp[decomp['time_series_type'] == 'history']
print(hist.tail(5))

print('''\nHow to read this for DocuMind:
  trend           -> overall growth in usage over the 30 days
  daily_pattern   -> peaks during business hours, troughs overnight
  weekly_pattern  -> dips on weekends (fewer documents processed)
  spikes_and_dips -> one-off events cleaned out of the baseline''')

## Exercise 7: Multi-Series Forecast

**Difficulty:** Challenge

Train with time_series_id_col. Forecast per document type. Compare patterns.

1. Add document_type as time_series_id_col
2. ML.FORECAST returns rows per type
3. Compare forecast shapes across types

In [ ]:
# Build a per-document-type table so each type is its own series.
# ARIMA_PLUS trains + forecasts each series independently in one model.
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.hourly_by_type` AS
WITH hours AS (
  SELECT ts
  FROM UNNEST(GENERATE_TIMESTAMP_ARRAY(
    TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY),
    CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)) AS ts
),
types AS (
  SELECT * FROM UNNEST(['invoice', 'contract', 'report']) AS document_type
)
SELECT
  ts AS hour_bucket,
  document_type,
  CAST(
    CASE document_type
      WHEN 'invoice'  THEN 40 + 25 * SIN(2 * ACOS(-1) * EXTRACT(HOUR FROM ts) / 24)
      WHEN 'contract' THEN 20 + 10 * SIN(2 * ACOS(-1) * EXTRACT(HOUR FROM ts) / 24)
      ELSE                 15 +  5 * SIN(2 * ACOS(-1) * EXTRACT(HOUR FROM ts) / 24)
    END
    + 15 * IF(EXTRACT(DAYOFWEEK FROM ts) IN (1, 7), -1, 1)
    + 8 * (RAND() - 0.5)
  AS INT64) AS query_count
FROM hours CROSS JOIN types
''')
print('Per-type table generated')

run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.query_forecast_by_type`
OPTIONS (
  model_type = 'ARIMA_PLUS',
  time_series_timestamp_col = 'hour_bucket',
  time_series_data_col = 'query_count',
  time_series_id_col = 'document_type',
  auto_arima = TRUE,
  data_frequency = 'HOURLY',
  forecast_limit_lower_bound = 0
) AS
SELECT hour_bucket, document_type, query_count
FROM `{PROJECT_ID}.rag_data.hourly_by_type`
''')
print('Multi-series model trained')

# ML.FORECAST now returns document_type as an extra column — one forecast per type
multi = run_query(f'''
SELECT
  document_type,
  forecast_timestamp,
  ROUND(forecast_value, 1) AS predicted_queries
FROM ML.FORECAST(
  MODEL `{PROJECT_ID}.ml_models.query_forecast_by_type`,
  STRUCT(24 AS horizon, 0.95 AS confidence_level))
ORDER BY document_type, forecast_timestamp
''')
print(f'\nForecast rows across all types: {len(multi)}')
print(multi.groupby('document_type')['predicted_queries'].agg(['mean', 'max']))

## Exercise 8: 3-Model Ops Dashboard

**Difficulty:** Challenge

Build all 3 operational queries: capacity forecast, cost anomalies, pattern explanation.

1. Forecast: hours exceeding threshold
2. Anomalies: cost spikes above 0.95 probability
3. Explain: trend + seasonality decomposition

In [ ]:
# Three operational views from the two ARIMA_PLUS models trained above.
print('=== 1. CAPACITY PLANNING ===')
print(run_query(f'''
SELECT forecast_timestamp, ROUND(forecast_value) AS predicted,
  ROUND(prediction_interval_upper_bound) AS worst_case
FROM ML.FORECAST(MODEL `{PROJECT_ID}.ml_models.query_forecast`, STRUCT(24 AS horizon))
WHERE prediction_interval_upper_bound > 80
'''))

print('\n=== 2. COST ANOMALIES ===')
print(run_query(f'''
SELECT hour_bucket, hourly_cost, ROUND(anomaly_probability, 3) AS prob
FROM ML.DETECT_ANOMALIES(
  MODEL `{PROJECT_ID}.ml_models.cost_anomaly`, STRUCT(0.95 AS anomaly_prob_threshold))
WHERE is_anomaly = TRUE ORDER BY anomaly_probability DESC LIMIT 5
'''))

print('\n=== 3. PATTERN EXPLANATION ===')
print(run_query(f'''
SELECT time_series_timestamp, ROUND(trend, 1) AS trend,
  ROUND(seasonal_period_daily, 1) AS daily, ROUND(seasonal_period_weekly, 1) AS weekly
FROM ML.EXPLAIN_FORECAST(
  MODEL `{PROJECT_ID}.ml_models.query_forecast`, STRUCT(24 AS horizon))
WHERE time_series_type = 'forecast' LIMIT 10
'''))